# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# Loading document by langChain's pdf loader

from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))
print(docs[0].metadata)

13
{'producer': 'Acrobat Distiller 5.0.5 for Macintosh (via http://big.faceless.org/products/pdf?version=2.8.3)', 'creator': 'FrameMaker 7.0', 'creationdate': '2004-12-13T15:22:54+00:00', 'author': 'DWest', 'moddate': '2014-10-24T15:09:14-06:00', 'title': 'R0501K_pdf.fm', 'source': 'https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1'}


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from typing import Literal
from pydantic import BaseModel, Field
from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader

# Pydantic models

class ArticleAnalysisLLM(BaseModel):
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(
        description="Why this article is relevant for an AI professional's professional development. Max one paragraph."
    )
    Summary: str = Field(
        description="Concise summary of the article, no longer than 1000 tokens."
    )
    Tone: str = Field(
        description="The clearly identifiable tone/style used to produce the summary."
    )


class ArticleAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [4]:
#2: Load the PDF and prepare dynamic context


file_path = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

loader = PyPDFLoader(file_path)
docs = loader.load()

print("Number of pages loaded:", len(docs))

context = "\n\n".join(doc.page_content for doc in docs)

print("First 500 characters of extracted context:\n")
print(context[:500])

Number of pages loaded: 13
First 500 characters of extracted context:

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


In [5]:
#3: Create separate developer instructions and user prompt


tone_style = "Formal Academic Writing"

developer_prompt = f"""
You are an expert reading assistant for professional development.

Return a structured response matching the provided schema exactly.

Requirements:
- Extract the article Author and Title.
- Write Relevance as no more than one paragraph.
- Write Summary as concise and succinct, no longer than 1000 tokens.
- Write the Summary in this specific tone: {tone_style}.
- Set Tone to the exact tone name used.
- Do not invent facts.
- Use only the provided article text.
""".strip()

user_prompt_template = """
Analyze the following article content and produce the structured output.

ARTICLE CONTENT:
{context}
""".strip()

user_prompt = user_prompt_template.format(context=context)

print("Developer prompt:\n")
print(developer_prompt)

print("\n" + "="*60 + "\n")

print("User prompt preview (first 700 chars):\n")
print(user_prompt[:700])

Developer prompt:

You are an expert reading assistant for professional development.

Return a structured response matching the provided schema exactly.

Requirements:
- Extract the article Author and Title.
- Write Relevance as no more than one paragraph.
- Write Summary as concise and succinct, no longer than 1000 tokens.
- Write the Summary in this specific tone: Formal Academic Writing.
- Set Tone to the exact tone name used.
- Do not invent facts.
- Use only the provided article text.


User prompt preview (first 700 chars):

Analyze the following article content and produce the structured output.

ARTICLE CONTENT:
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s idea

In [6]:
import sys
sys.path.append('../../05_src')

In [7]:
from dotenv import load_dotenv
load_dotenv('../05_src/.secrets')

True

In [10]:
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import os

# Remove the bad value from the current kernel, if present
if os.getenv("OPENAI_API_KEY") == "any_value":
    del os.environ["OPENAI_API_KEY"]

# Load the real key from your file
dotenv_path = Path("../05_src/.secrets").resolve()
print("Using:", dotenv_path)
print("Exists?", dotenv_path.exists())

loaded = load_dotenv(dotenv_path=dotenv_path, override=True)
print("load_dotenv returned:", loaded)

api_key = os.getenv("OPENAI_API_KEY")
print("Loaded preview:", None if not api_key else api_key[:10])

if not api_key or api_key == "any_value":
    raise ValueError("OPENAI_API_KEY is still missing or invalid in this kernel.")

client = OpenAI(api_key=api_key)
print("Client created successfully.")

Using: C:\Users\skiev\UoTAI\deploying-ai\05_src\.secrets
Exists? True
load_dotenv returned: True
Loaded preview: sk-proj-Sr
Client created successfully.


In [12]:
#4: Call the OpenAI API with structured output

from pathlib import Path
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.parse(
    model="gpt-4.1-mini",
    messages=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": user_prompt},
    ],
    response_format=ArticleAnalysisLLM,
)

parsed = completion.choices[0].message.parsed

print("Structured model output from the LLM:\n")
print(parsed.model_dump())

print("\nToken usage:")
print("Input tokens:", completion.usage.prompt_tokens)
print("Output tokens:", completion.usage.completion_tokens)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [13]:
# =========================
# SECTION 4: Call the OpenAI API with structured output
# using API_GATEWAY_KEY
# =========================

from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import os

# Load secrets file
dotenv_path = Path("../05_src/.secrets").resolve()

if not dotenv_path.exists():
    raise FileNotFoundError(f".secrets file not found at: {dotenv_path}")

load_dotenv(dotenv_path=dotenv_path, override=True)

# Read the gateway key instead of OPENAI_API_KEY
api_gateway_key = os.getenv("API_GATEWAY_KEY")
print("API_GATEWAY_KEY loaded?", api_gateway_key is not None)
print("Preview:", None if not api_gateway_key else api_gateway_key[:10])

if not api_gateway_key or api_gateway_key == "any_value":
    raise ValueError("API_GATEWAY_KEY is missing or invalid in .secrets")

# If your gateway also requires a custom base URL, store it in .secrets too
api_base_url = os.getenv("API_BASE_URL")

if api_base_url:
    client = OpenAI(
        api_key=api_gateway_key,
        base_url=api_base_url
    )
else:
    client = OpenAI(
        api_key=api_gateway_key
    )

completion = client.chat.completions.parse(
    model="gpt-4.1-mini",
    messages=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": user_prompt},
    ],
    response_format=ArticleAnalysisLLM,
)

parsed = completion.choices[0].message.parsed

print("Structured model output from the LLM:\n")
print(parsed.model_dump())

print("\nToken usage:")
print("Input tokens:", completion.usage.prompt_tokens)
print("Output tokens:", completion.usage.completion_tokens)

API_GATEWAY_KEY loaded? True
Preview: AdPTunl2po


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: AdPTunl2********************97wI. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [ ]:
#5: Build the final Pydantic object with token counts

final_result = ArticleAnalysis(
    Author=parsed.Author,
    Title=parsed.Title,
    Relevance=parsed.Relevance,
    Summary=parsed.Summary,
    Tone=parsed.Tone,
    InputTokens=completion.usage.prompt_tokens,
    OutputTokens=completion.usage.completion_tokens,
)

print("Final structured output:\n")
print(final_result.model_dump_json(indent=2))

NameError: name 'parsed' is not defined

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
